<a href="https://colab.research.google.com/github/serenawong9904-lgtm/cv-marketing-campaign-visual-audit-bot/blob/feature_ocr_pipeline/WID3013_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install easyocr

In [2]:
# GENERATE A MOCK DIGITAL POSTER FOR PIPELINE VALIDATION
import cv2
import numpy as np

# Create a blank white canvas representing a typical marketing flyer layout (Height: 800px, Width: 600px)
mock_poster = np.ones((800, 600, 3), dtype=np.uint8) * 255

# Inject common marketing text structures to mimic a real student event poster layout
cv2.putText(mock_poster, "ANNUAL MARKETING SUMMIT 2026", (50, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (20, 20, 20), 2)
cv2.putText(mock_poster, "Organized by the Business Club", (50, 180), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (100, 100, 100), 1)
cv2.putText(mock_poster, "Gain insights from industrial leaders and professional marketers.", (50, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (50, 50, 50), 1)

# Add clear Call-to-Action phrases to test your regex matching logic
cv2.putText(mock_poster, "Scan Here to Register", (160, 650), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 0, 0), 2)
cv2.putText(mock_poster, "Join Now", (250, 720), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 150, 0), 2)

# Save the mock asset to local disk
cv2.imwrite("user_poster.jpg", mock_poster)
print("Success: 'user_poster.jpg' generated and saved to your local Colab environment workspace.")

Success: 'user_poster.jpg' generated and saved to your local Colab environment workspace.


In [3]:
# GENERALIZED TEXT REGION DETECTION & SPATIAL MAPPING PIPELINE
import re
import json
import easyocr

def extract_poster_text_and_coordinates(image_path="user_poster.jpg"):

    print(f"Initializing Adaptive EasyOCR Extraction Pipeline for: {image_path}")

    # Initialize the reader for English language processing
    reader = easyocr.Reader(['en'], gpu=False)
    results = reader.readtext(image_path)

    if not results:
        return {
            "status": "error",
            "message": "No text components or visual layouts detected in the image matrix."
        }

    all_extracted_elements = []
    headline_text = ""
    max_bounding_area = 0

    # Generalized marketing pattern dictionary applicable across all domains
    # (Events, Products, Retail, Restaurants)
    cta_patterns = [
        r"join\s*now", r"register\s*now", r"scan\s*here", r"scan\s*me",
        r"apply\s*now", r"book\s*now", r"rsvp", r"buy\s*now", r"order\s*now",
        r"click\s*here", r"visit\s*us", r"get\s*yours", r"limited\s*offer"
    ]
    cta_regex = re.compile("|".join(cta_patterns), re.IGNORECASE)
    detected_ctas = []

    # Iterate through the structural geometry outputs provided by the engine
    for (bbox, text, confidence) in results:
        # FIXED BUG: Correctly indexing EasyOCR's bounding box matrix
        # [[x0, y0], [x1, y1], [x2, y2], [x3, y3]] -> Top-Left and Bottom-Right
        top_left = [int(bbox[0][0]), int(bbox[0][1])]
        bottom_right = [int(bbox[2][0]), int(bbox[2][1])]

        cleaned_text = text.strip()

        # FIXED BUG: Extract integer values out of the coordinate lists before math operations
        width = bottom_right[0] - top_left[0]
        height = bottom_right[1] - top_left[1]
        current_area = width * height

        element_entry = {
            "text": cleaned_text,
            "confidence": float(round(confidence, 3)),
            "bounding_box": {
                "top_left": top_left,
                "bottom_right": bottom_right
            }
        }
        all_extracted_elements.append(element_entry)

        # HEADLINE HEURISTIC: Mathematically evaluates visual prominence.
        # Bypasses layout boundaries or system frames containing 'preview' or '.jpg'
        if current_area > max_bounding_area and len(cleaned_text) > 3:
            if not any(ext in cleaned_text.lower() for ext in ["preview", ".jpg", ".png", ".jpeg"]):
                max_bounding_area = current_area
                headline_text = cleaned_text

        # CALL TO ACTION HEURISTIC: Universal phrase pattern match
        if cta_regex.search(cleaned_text):
            detected_ctas.append({
                "text": cleaned_text,
                "type": "textual_intent",
                "coordinates": {
                    "top_left": top_left,
                    "bottom_right": bottom_right
                }
            })

    # Return structured payload data contract to hand over to Member 3
    return {
        "status": "success",
        "metadata": {
            "total_text_regions_found": len(all_extracted_elements)
        },
        "extracted_content": {
            "headline": headline_text if headline_text else "None Detected Confidently",
            "detected_call_to_actions": detected_ctas,
            "raw_text_stream": [item["text"] for item in all_extracted_elements],
            "complete_spatial_manifest": all_extracted_elements
        }
    }

# --- PIPELINE INTEGRATION TESTING ---
# Synchronized with Member 1's automated image downloader destination path
if __name__ == "__main__":
    try:
        data_bridge_payload = extract_poster_text_and_coordinates("user_poster.jpg")
        print("\n=== DATA CONTRACT OUTPUT FOR MEMBER 3 ===")
        print(json.dumps(data_bridge_payload, indent=4))
    except Exception as e:
        print(f"Execution Error: {str(e)}")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\User\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.